In [0]:
%run ../00_common/data_utils

In [0]:
def generate_rebind_ukey(task_id):

    # 包含 ACS和LineBind 源系统
    acs_lind_source_df = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config")
        .filter(F.col("tmec_type").isin(SOURCE_TYPE_ACS, SOURCE_TYPE_LINEBIND, RAKUTEN, LINEGIFT))
        .select(
            F.col("tmec_marketcode").alias("exclude_mrkt"),
            F.col("tmec_sourcesystemcode").alias("exclude_srcs_code"),
        )
        .distinct()
    )

    new_ukey_df = (get_ukey_group_by_process(task_id)
            .filter(F.col("is_master_recode") == False)
            .select("task_id", "mrkt_code", "srcc_id", "new_consumermdmkey")
            .distinct())

    # Regular consumer
    clean_consumer = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer").alias("tcc")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("is_include") == True)
        .join(acs_lind_source_df,
            (F.col("srcc_srcs_code") == F.col("exclude_srcs_code")) &
            (F.col("srcc_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti"
        )
        .join(new_ukey_df.alias("nud"), (F.col("tcc.srcc_mrkt_code") == F.col("nud.mrkt_code")) & (F.col("tcc.srcc_id") == F.col("nud.srcc_id")), "left")
        .select(
            "tcc.*",
            "nud.new_consumermdmkey"
        )
    )


    # 筛选 linedind中没有uid的master consumer
    master_consumer = (spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
        .join(
            acs_lind_source_df.filter(F.col("tmec_type") == "LineBind"),
            (F.col("SCON_MRKT_CODE") == F.col("exclude_mrkt")) & (F.col("scon_srcs_code") == F.col("exclude_srcs_code")),
            "inner"
        )
        .filter(F.coalesce(F.col("consumermdmkey"), F.lit("")) == "")
        .select(
            F.col("SCON_ID"),
            F.col("SCON_SRCC_ID"),
            F.col("SCON_MRKT_CODE"),
            F.col("SCON_BRND_CODE"),
            F.col("SCON_SRCS_CODE"),
            F.col("SCON_CONSUMERID"),
            F.col("SCON_MASTERCONSUMERID"),
            F.col("SCON_SOURCETIMESTAMP")
        )
    )


    # linebind consumer进行uid重绑定
    line_rebind_df = (
    master_consumer.alias("m")
        .join(
            clean_consumer.alias("c"),
            (F.col("m.SCON_MRKT_CODE") == F.col("c.SRCC_MRKT_CODE"))  & (F.col("m.SCON_BRND_CODE") == F.col("c.SRCC_BRND_CODE")) & (F.col("m.SCON_MASTERCONSUMERID") == F.col("c.SRCC_CONSUMERID")),
            "inner"
        )
        .withColumn("rk", F.row_number().over(Window.partitionBy("m.SCON_MRKT_CODE", "m.SCON_SRCC_ID").orderBy(
                F.col("c.SRCC_SOURCETIMESTAMP").desc(),
                F.col("c.SRCC_ID").desc()
            )
        ))
        .filter(F.col("rk") == 1)
        .select(
            F.expr("uuid()").alias("matc_id"),
            F.col("m.SCON_SRCC_ID").alias("srcc_id"),
            F.col("m.SCON_MRKT_CODE").alias("mrkt_code"),
            F.col("m.SCON_BRND_CODE").alias("brnd_code"),
            F.col("m.SCON_SRCS_CODE").alias("source_code"),
            F.col("m.SCON_CONSUMERID").alias("consumer_id"),
            F.col("m.SCON_SOURCETIMESTAMP").alias("source_timestamp"),
            F.col("c.new_consumermdmkey"),
            F.col("m.SCON_ID").alias("master_scon_id"),
            F.lit(None).alias("master_consumermdmkey"),
            F.lit(None).alias("master_recode_create_time"),
            F.lit(True).alias("is_master_recode"),
            F.lit(None).alias("gid"),
            F.lit(None).alias("grp_size"),
            F.lit(None).alias("batch_id"),
            F.lit(task_id).alias("task_id"),
            F.lit(MATCH_TYPE_REBIND_STR).alias("match_type"),
            F.current_timestamp().alias("creation_dt")   
        )
        .distinct()
    )

    save_to_target_table(
        line_rebind_df,
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group",
        f"task_id = '{task_id}' and match_type = '{MATCH_TYPE_REBIND_STR}' "
    )

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")


with StepLogger("4.3_ukey_match_rebind", "04-3", "consumerlist", task_id=task_id) as logger:
    generate_rebind_ukey(task_id)